# PyRosetta Scoring Functions

This notebook demonstrates the three PyRosetta scoring operations wrapped in `proto_tools`: Spatial Aggregation Propensity (SAP), Solvent Accessible Surface Area (SASA), and full Rosetta energy scoring. Each tool takes one or more protein structures and returns physics-based numeric scores with per-residue breakdowns where applicable.

**License:** PyRosetta requires acceptance of the [Rosetta Software License](https://www.rosettacommons.org/software/license-and-download). Free for academic and non-commercial use; commercial users must obtain a separate license.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pyrosetta")
display_overview("pyrosetta")
display_docs_section("pyrosetta", "Background")

## Available tools

In [ ]:
display_available_tools("pyrosetta")

### `run_pyrosetta_sap`

SAP (Spatial Aggregation Propensity) quantifies how much hydrophobic surface area is exposed on a protein. Proteins with large patches of exposed hydrophobicity are prone to aggregation — a major concern in therapeutic protein development. `run_pyrosetta_sap` loads each input structure into a Rosetta Pose and scores it with Rosetta's `core.pack.guidance_scoreterms.sap` module, returning a single scalar SAP score per structure. Higher SAP scores indicate greater aggregation risk.

In [ ]:
from proto_tools import (
    PyRosettaSAPConfig,
    PyRosettaSAPInput,
    run_pyrosetta_sap,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sap import example_input as sap_example_input

In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sap")

# Use the tool's canonical example input PDB.
sap_inputs = sap_example_input()

# Let's take a look at what this looks like
sap_inputs.inputs[0].structure.visualize(color_by="chain")

In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sap")

# PyRosettaSAPConfig has no tool-specific parameters; all defaults are used
sap_config = PyRosettaSAPConfig()

In [ ]:
# Run the SAP scoring function
sap_result = run_pyrosetta_sap(sap_inputs, sap_config)

In [ ]:
# Display output docs and inspect results
display_api_reference("pyrosetta", "output", "run_pyrosetta_sap")

for r in sap_result.results:
    print(f"SAP score: {r.sap_score:.2f}")

### `run_pyrosetta_sasa`

SASA (Solvent Accessible Surface Area) measures the total protein surface accessible to a spherical solvent probe (1.4 A by default, matching water). Buried residues contribute to the hydrophobic core, while exposed residues interact with solvent and binding partners. `run_pyrosetta_sasa` returns both the total SASA and a per-residue breakdown, which is useful for identifying surface-exposed sites, characterizing the hydrophobic core, or feeding into downstream feature engineering.

In [ ]:
from proto_tools import (
    PyRosettaSASAConfig,
    PyRosettaSASAInput,
    run_pyrosetta_sasa,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_sasa import example_input as sasa_example_input

In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_sasa")

sasa_inputs = sasa_example_input()

In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_sasa")

# 1.4 A is the standard water probe radius
sasa_config = PyRosettaSASAConfig(probe_radius=1.4)

In [ ]:
# Run the SASA computation
sasa_result = run_pyrosetta_sasa(sasa_inputs, sasa_config)

In [ ]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_sasa")

# Total SASA plus the five most solvent-exposed residues
r = sasa_result.results[0]
print(f"Total SASA: {r.total_sasa:.1f} A^2")
print(f"Residues:   {len(r.per_residue)}")

top_exposed = sorted(r.per_residue, key=lambda res: res.sasa, reverse=True)[:5]
print("\nTop 5 most exposed residues:")
for res in top_exposed:
    print(f"  {res.chain_id}:{res.residue_name}{res.residue_index} — {res.sasa:.1f} A^2")

### `run_pyrosetta_energy`

Rosetta energy scoring evaluates a protein structure with a physics-based energy function that combines van der Waals interactions, electrostatics, hydrogen bonding, solvation, and backbone torsion terms. The score is reported in Rosetta Energy Units (REU); lower (more negative) is more favorable. `run_pyrosetta_energy` optionally runs [FastRelax](https://www.rosettacommons.org/docs/latest/scripting_documentation/RosettaScripts/Movers/movers_pages/FastRelaxMover) before scoring to resolve clashes and backbone strain, and returns the total energy, a per-term breakdown, and per-residue totals. The default score function is `ref2015`, Rosetta's current production all-atom energy function.

In [ ]:
from proto_tools import (
    PyRosettaEnergyConfig,
    PyRosettaEnergyInput,
    run_pyrosetta_energy,
)
from proto_tools.tools.structure_scoring.pyrosetta.pyrosetta_energy import example_input as energy_example_input

In [ ]:
# Display input docs
display_api_reference("pyrosetta", "input", "run_pyrosetta_energy")

energy_inputs = energy_example_input()

In [ ]:
# Display config docs
display_api_reference("pyrosetta", "config", "run_pyrosetta_energy")

# Use ref2015 with a single FastRelax cycle to keep the example fast
energy_config = PyRosettaEnergyConfig(
    scorefxn="ref2015",
    relax=True,
    relax_cycles=1,
)

In [ ]:
# Run the energy scoring function
energy_result = run_pyrosetta_energy(energy_inputs, energy_config)

In [ ]:
# Display output docs
display_api_reference("pyrosetta", "output", "run_pyrosetta_energy")

r = energy_result.results[0]
print(f"Total energy: {r.total_energy:.1f} REU")
print(f"Relaxed:      {r.relaxed}")
print("\nTop contributing energy terms:")
for term, value in sorted(r.energy_terms.items(), key=lambda kv: kv[1])[:6]:
    print(f"  {term:>12s}: {value:.2f}")

## Export Results

Each output model supports export to common file formats for downstream analysis. SASA results export well to CSV because of the per-residue table; energy results export to JSON to preserve the nested per-term breakdown.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sap_result.export("pyrosetta_sap", export_path=output_dir, file_format="json")
sasa_result.export("pyrosetta_sasa", export_path=output_dir, file_format="csv")
energy_result.export("pyrosetta_energy", export_path=output_dir, file_format="json")

for name in ("pyrosetta_sap.json", "pyrosetta_sasa.csv", "pyrosetta_energy.json"):
    print(f"Exported {output_dir / name}")